# 05 — Agent Routing

Wire the 5 tools into a LangGraph ReAct agent. Test routing accuracy, multi-turn conversation, and follow-up questions.

**Key challenge:** Tool descriptions must be precise enough for Claude to route correctly:
- "What is this about?" → summarize_video
- "What does it say about X?" → vector_search
- "What topics are covered?" → get_topics
- "Compare videos on X" → compare_videos
- "How long is this video?" → get_metadata

In [1]:
import os
import json
from dotenv import load_dotenv
from pinecone import Pinecone
from anthropic import Anthropic
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# Clients (module-level, used by tool closures)
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pine_index = pc.Index(os.getenv("PINECONE_INDEX_NAME", "askthevideo"))
anthropic_client = Anthropic()

CLAUDE_MODEL = "claude-sonnet-4-6"
VIDEO_ID = "e-gwvmhyU7A"

# Simulated session state: which videos are currently loaded
loaded_videos = [VIDEO_ID]

print(f"Clients ready. Loaded videos: {loaded_videos}")

Clients ready. Loaded videos: ['e-gwvmhyU7A']


In [2]:
EMBED_MODEL = "llama-text-embed-v2"
SENTINEL_VECTOR = [1e-7] + [0.0] * 1023

def _embed_texts(texts, input_type="passage"):
    all_embs = []
    for i in range(0, len(texts), 50):
        batch = texts[i:i+50]
        embs = pc.inference.embed(
            model=EMBED_MODEL, inputs=batch,
            parameters={"input_type": input_type, "truncate": "END"},
        )
        all_embs.extend([e.values for e in embs])
    return all_embs

def _query_chunks(question, video_id, top_k=5):
    emb = _embed_texts([question], input_type="query")[0]
    results = pine_index.query(
        vector=emb, namespace=video_id,
        top_k=top_k, include_metadata=True,
    )
    return [
        {"score": m.score, "id": m.id, **m.metadata}
        for m in results.matches
        if m.metadata.get("type") != "metadata"
    ]

def _fetch_all_chunks(video_id):
    stats = pine_index.describe_index_stats()
    total = int(stats.namespaces.get(video_id, {}).get("vector_count", 0))
    chunk_ids = [f"{video_id}_chunk_{i:03d}" for i in range(total)]
    fetched = pine_index.fetch(ids=chunk_ids, namespace=video_id)
    chunks = []
    for cid in chunk_ids:
        vec = fetched.vectors.get(cid)
        if vec and vec.metadata.get("type", "chunk") == "chunk":
            chunks.append(dict(vec.metadata))
    chunks.sort(key=lambda c: c.get("chunk_index", 0))
    return chunks

def _build_full_text(chunks):
    return "\n\n".join(
        f"[{c['start_display']}–{c['end_display']}]\n{c['text']}"
        for c in chunks
    )

print("Helpers loaded")

Helpers loaded


In [3]:
@tool
def vector_search(question: str) -> str:
    """Search the loaded video transcripts to answer a specific question about the content.
    Use this when the user asks about something specific mentioned in a video,
    wants details about a particular topic discussed, or asks a factual question
    about what was said. Do NOT use for summaries or topic lists."""

    all_chunks = []
    per_video_k = max(3, 10 // len(loaded_videos))
    for vid in loaded_videos:
        all_chunks.extend(_query_chunks(question, vid, top_k=per_video_k))

    if not all_chunks:
        return "No relevant content found in the loaded videos."

    all_chunks.sort(key=lambda c: c["score"], reverse=True)
    all_chunks = all_chunks[:10]

    context_parts = []
    for c in all_chunks:
        header = f"[{c['start_display']}–{c['end_display']}] ({c['video_url']})"
        context_parts.append(f"{header}\n{c['text_timestamped']}")
    context = "\n\n---\n\n".join(context_parts)

    response = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system="You answer questions about YouTube videos using transcript excerpts. "
               "Always reference specific timestamps from the excerpts. "
               "If the excerpts don't contain relevant information, say so.",
        messages=[{
            "role": "user",
            "content": f"Transcript excerpts:\n\n{context}\n\n---\n\nQuestion: {question}",
        }],
    )
    return response.content[0].text


@tool
def summarize_video(video_id: str) -> str:
    """Generate a summary of a specific video. Use this when the user asks
    'what is this video about?', 'summarize this video', 'give me an overview',
    or similar broad questions about the whole video content.
    The video_id parameter should be taken from the loaded videos list."""

    record_id = f"{video_id}_summary"
    cached = pine_index.fetch(ids=[record_id], namespace=video_id)
    vec = cached.vectors.get(record_id)
    if vec and vec.metadata.get("text"):
        return vec.metadata["text"]

    chunks = _fetch_all_chunks(video_id)
    if not chunks:
        return "No transcript data found for this video."

    full_text = _build_full_text(chunks)
    response = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2048,
        system="You summarise YouTube videos from their transcript. "
               "Provide: a one-paragraph overview, then 5-7 key points with timestamps. "
               "Be concise and specific.",
        messages=[{"role": "user", "content": f"Summarise this video transcript:\n\n{full_text}"}],
    )

    result_text = response.content[0].text
    pine_index.upsert(
        vectors=[{
            "id": record_id,
            "values": SENTINEL_VECTOR,
            "metadata": {"type": "summary", "video_id": video_id, "text": result_text},
        }],
        namespace=video_id,
    )
    return result_text


@tool
def list_topics(video_id: str) -> str:
    """List the main topics covered in a video with timestamps.
    Use this when the user asks 'what topics are covered?', 'give me an outline',
    'what does the video talk about?' or wants a table of contents.
    The video_id parameter should be taken from the loaded videos list."""

    record_id = f"{video_id}_topics"
    cached = pine_index.fetch(ids=[record_id], namespace=video_id)
    vec = cached.vectors.get(record_id)
    if vec and vec.metadata.get("text"):
        return vec.metadata["text"]

    chunks = _fetch_all_chunks(video_id)
    if not chunks:
        return "No transcript data found for this video."

    full_text = _build_full_text(chunks)
    response = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system="You extract the main topics from a YouTube video transcript. "
               "Return a numbered list of 8-12 topics, each with a timestamp range "
               "and a one-sentence description. Format: '1. [MM:SS-MM:SS] Topic — description'",
        messages=[{"role": "user", "content": f"Extract the main topics from this transcript:\n\n{full_text}"}],
    )

    result_text = response.content[0].text
    pine_index.upsert(
        vectors=[{
            "id": record_id,
            "values": SENTINEL_VECTOR,
            "metadata": {"type": "topics", "video_id": video_id, "text": result_text},
        }],
        namespace=video_id,
    )
    return result_text


@tool
def compare_videos(question: str) -> str:
    """Compare what multiple loaded videos say about a specific topic.
    Use this ONLY when the user explicitly asks to compare videos or asks
    what different videos say about the same topic. Requires 2+ loaded videos
    to be meaningful, but works with 1 video as a focused topic summary."""

    all_chunks = []
    per_video_k = max(3, 10 // len(loaded_videos))
    for vid in loaded_videos:
        chunks = _query_chunks(question, vid, top_k=per_video_k)
        for c in chunks:
            c["video_id"] = vid
        all_chunks.extend(chunks)

    if not all_chunks:
        return "No relevant content found."

    all_chunks.sort(key=lambda c: c["score"], reverse=True)

    by_video = {}
    for c in all_chunks:
        vid = c["video_id"]
        if vid not in by_video:
            by_video[vid] = []
        by_video[vid].append(c)

    context_parts = []
    for vid, chunks in by_video.items():
        header = f"=== Video: {vid} ==="
        excerpts = [
            f"[{c['start_display']}–{c['end_display']}] ({c['video_url']})\n{c['text_timestamped']}"
            for c in chunks
        ]
        context_parts.append(header + "\n\n" + "\n\n".join(excerpts))

    context = ("\n\n" + "=" * 40 + "\n\n").join(context_parts)

    response = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1536,
        system="You compare what different YouTube videos say about a topic. "
               "Highlight similarities and differences. Reference specific timestamps. "
               "If only one video is provided, summarise what it says about the topic.",
        messages=[{
            "role": "user",
            "content": f"Compare these videos on the topic:\n\nQuestion: {question}\n\n{context}",
        }],
    )
    return response.content[0].text


@tool
def get_metadata(video_id: str) -> str:
    """Get metadata about a loaded video: title, channel, duration, chunk count.
    Use this when the user asks 'what video is loaded?', 'who made this?',
    'how long is the video?', or wants basic video information.
    The video_id parameter should be taken from the loaded videos list."""

    result = pine_index.fetch(ids=[f"{video_id}_metadata"], namespace=video_id)
    vec = result.vectors.get(f"{video_id}_metadata")
    if vec:
        meta = vec.metadata
        return (
            f"Video: {meta.get('video_title', 'Unknown')}\n"
            f"Channel: {meta.get('channel', 'Unknown')}\n"
            f"Duration: {meta.get('duration_display', 'Unknown')}\n"
            f"Chunks: {int(meta.get('chunk_count', 0))}\n"
            f"Ingested: {meta.get('ingested_at', 'Unknown')}"
        )
    return "Metadata not found for this video."


tools = [vector_search, summarize_video, list_topics, compare_videos, get_metadata]
print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

Defined 5 tools: ['vector_search', 'summarize_video', 'list_topics', 'compare_videos', 'get_metadata']


In [4]:
llm = ChatAnthropic(model=CLAUDE_MODEL, temperature=0.0)
memory = MemorySaver()

SYSTEM_PROMPT = (
    "You are AskTheVideo, an AI assistant that answers questions about YouTube videos. "
    f"Currently loaded videos: {loaded_videos}\n\n"
    "Rules:\n"
    "- Use the available tools to answer questions. Do not make up information.\n"
    "- For specific questions about video content, use vector_search.\n"
    "- For 'what is this about' or 'summarize', use summarize_video with the video_id.\n"
    "- For 'what topics' or 'outline', use list_topics with the video_id.\n"
    "- For comparisons across videos, use compare_videos.\n"
    "- For video info (title, duration, channel), use get_metadata.\n"
    "- Always pass video_id as a string, not a list.\n"
    "- Include timestamp links in your answers when available."
)

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT, checkpointer=memory)
print("Agent created")

Agent created


/var/folders/wc/70rj4v0n31dctgshd9k1btcm0000gp/T/ipykernel_70153/3552870736.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT, checkpointer=memory)


In [5]:
import time

test_queries = [
    ("What do they say about Perplexity AI?", "vector_search"),
    ("Summarize this video", "summarize_video"),
    ("What topics are covered?", "list_topics"),
    ("Compare what the videos say about transformers", "compare_videos"),
    ("How long is this video?", "get_metadata"),
    ("What is this video about?", "summarize_video"),
    ("Who is the guest on this podcast?", "vector_search"),
    ("Give me an outline of the video", "list_topics"),
    ("What does the speaker think about Google?", "vector_search"),
    ("What video is loaded?", "get_metadata"),
]

config = {"configurable": {"thread_id": "routing_test"}}
correct = 0

for question, expected_tool in test_queries:
    start = time.time()
    result = agent.invoke({"messages": [("user", question)]}, config)

    # Find which tool was called (look at message types)
    tool_called = None
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_called = msg.tool_calls[0]["name"]
            break

    elapsed = time.time() - start
    match = "✅" if tool_called == expected_tool else "❌"
    if tool_called == expected_tool:
        correct += 1
    print(f"  {match} [{elapsed:5.1f}s] '{question[:50]}' → {tool_called} (expected: {expected_tool})")

    # Reset thread to avoid multi-turn context leaking between tests
    config = {"configurable": {"thread_id": f"routing_test_{correct}"}}

print(f"\nRouting accuracy: {correct}/{len(test_queries)} ({correct/len(test_queries)*100:.0f}%)")

  ✅ [ 28.0s] 'What do they say about Perplexity AI?' → vector_search (expected: vector_search)
  ✅ [ 16.3s] 'Summarize this video' → summarize_video (expected: summarize_video)
  ✅ [  6.5s] 'What topics are covered?' → list_topics (expected: list_topics)
  ✅ [ 39.6s] 'Compare what the videos say about transformers' → compare_videos (expected: compare_videos)
  ✅ [  3.8s] 'How long is this video?' → get_metadata (expected: get_metadata)
  ✅ [  7.9s] 'What is this video about?' → summarize_video (expected: summarize_video)
  ✅ [  7.9s] 'Who is the guest on this podcast?' → vector_search (expected: vector_search)
  ✅ [ 11.5s] 'Give me an outline of the video' → list_topics (expected: list_topics)
  ✅ [ 21.9s] 'What does the speaker think about Google?' → vector_search (expected: vector_search)
  ✅ [  3.9s] 'What video is loaded?' → get_metadata (expected: get_metadata)

Routing accuracy: 10/10 (100%)


In [6]:
# Multi-turn conversation test
config = {"configurable": {"thread_id": "multiturn_test"}}

conversation = [
    "What is this video about?",
    "Tell me more about what they said about Google",
    "What topics are covered?",
]

for i, question in enumerate(conversation):
    start = time.time()
    result = agent.invoke({"messages": [("user", question)]}, config)

    # Get the final assistant message
    final_msg = result["messages"][-1].content
    elapsed = time.time() - start

    # Find tool used
    tool_called = None
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_called = msg.tool_calls[0]["name"]

    print(f"\n{'='*60}")
    print(f"Turn {i+1} [{elapsed:.1f}s] Tool: {tool_called}")
    print(f"Q: {question}")
    print(f"A: {final_msg[:300]}...")


Turn 1 [7.9s] Tool: summarize_video
Q: What is this video about?
A: This is an episode of the **Lex Fridman Podcast** featuring **Aravind Srinivas**, the CEO of **Perplexity AI**. It's a wide-ranging conversation covering:

- 🔍 **What Perplexity is** — an "answer engine" that combines web search with LLMs to deliver cited, source-backed answers (rather than just a l...

Turn 2 [26.8s] Tool: vector_search
Q: Tell me more about what they said about Google
A: Here's a detailed breakdown of what was discussed about Google:

---

### 💰 Google's Business Model ([18:03])
Aravind calls Google's AdWords system **"the greatest business model in the last 50 years."** Interestingly, he notes the model was *originally conceived by Overture*, but Google improved th...

Turn 3 [9.7s] Tool: list_topics
Q: What topics are covered?
A: Here's a full outline of the topics covered in the video:

| # | Timestamp | Topic |
|---|-----------|-------|
| 1 | [0:00] | **Introduction** — Lex introduces Perplexity

## Routing results

**Accuracy: 10/10 (100%)** — no description tuning needed.

| Query pattern | Tool selected | Correct |
|---|---|---|
| Specific content question (3x) | vector_search | ✅ |
| Summarize / what is this about (2x) | summarize_video | ✅ |
| Topics / outline (2x) | list_topics | ✅ |
| Compare videos (1x) | compare_videos | ✅ |
| Video info / metadata (2x) | get_metadata | ✅ |

**Multi-turn:** 3-turn conversation maintained context. Follow-up question "Tell me more about what they said about Google" correctly routed to vector_search using conversational context.

**LangSmith fix:** Initial 403 errors caused by hitting US endpoint (`api.smith.langchain.com`) while account is EU-hosted. Fixed by adding `LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com` to `.env`. Traces now visible at `eu.smith.langchain.com` under the `askthevideo` project. Decision #65 logged.

**Latency per turn:**

| Tool | Median | Notes |
|---|---|---|
| get_metadata | ~4s | No Claude call, Pinecone fetch only |
| list_topics (cached) | ~9s | Cache hit, agent overhead |
| summarize_video (cached) | ~8s | Cache hit, agent overhead |
| vector_search | ~20s | Embed + retrieve + Claude generation |
| compare_videos | ~40s | Multi-chunk retrieval + Claude generation |

Agent overhead adds ~3-5s on top of raw tool latency (LLM reasoning about which tool to call).

In [8]:
help(create_agent)

Help on function create_agent in module langchain.agents.factory:

create_agent(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'
    Creates an agent graph that calls tools in a loop until a stopping condition is met.

    For more details on using `cre

In [9]:
test_agent = create_agent(test_llm, tools, system_prompt=SYSTEM_PROMPT, checkpointer=test_memory)

config = {"configurable": {"thread_id": "import_test"}}
result = test_agent.invoke({"messages": [("user", "How long is this video?")]}, config)

tool_called = None
for msg in result["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tool_called = msg.tool_calls[0]["name"]

print(f"Tool: {tool_called}")
print(f"Answer: {result['messages'][-1].content[:200]}")
print("\n✅ New import works" if tool_called == "get_metadata" else "\n❌ Issue")

Tool: get_metadata
Answer: The video is **3 hours, 2 minutes, and 6 seconds** long. It's titled *"AI tools podcast (test video)"* from the **Test Channel**. Let me know if you have any questions about its content!

✅ New import works


## Production function

In [10]:
# @export src/agent.py
import os
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver


def create_askthevideo_agent(tools: list, loaded_videos: list[str]) -> tuple:
    """Create a LangGraph agent with the given tools.

    Args:
        tools: list of LangChain @tool functions
        loaded_videos: list of currently loaded video_ids

    Returns:
        (agent, memory) tuple
    """
    llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0.0)
    memory = MemorySaver()

    system_prompt = (
        "You are AskTheVideo, an AI assistant that answers questions about YouTube videos. "
        f"Currently loaded videos: {loaded_videos}\n\n"
        "Rules:\n"
        "- Use the available tools to answer questions. Do not make up information.\n"
        "- For specific questions about video content, use vector_search.\n"
        "- For 'what is this about' or 'summarize', use summarize_video with the video_id.\n"
        "- For 'what topics' or 'outline', use list_topics with the video_id.\n"
        "- For comparisons across videos, use compare_videos.\n"
        "- For video info (title, duration, channel), use get_metadata.\n"
        "- Always pass video_id as a string, not a list.\n"
        "- Include timestamp links in your answers when available."
    )

    agent = create_agent(llm, tools, system_prompt=system_prompt, checkpointer=memory)
    return agent, memory

## Summary

**What we built:**
- 5 LangChain `@tool` wrapped functions (vector_search, summarize_video, list_topics, compare_videos, get_metadata)
- `create_askthevideo_agent()` — factory function that builds a LangGraph agent with MemorySaver

**Production cells tagged:** 1 cell → `src/agent.py`

**Routing accuracy: 10/10 (100%)**
- Tool descriptions were precise enough on the first try — no tuning needed
- Follow-up questions route correctly using conversational context

**Multi-turn:** MemorySaver maintains conversation history across turns. Follow-up "Tell me more about what they said about Google" correctly routed to vector_search using context from previous summary turn.

**Latency per turn:**

| Tool | Median | Notes |
|---|---|---|
| get_metadata | ~4s | No Claude call, Pinecone fetch only |
| list_topics (cached) | ~9s | Cache hit + agent overhead |
| summarize_video (cached) | ~8s | Cache hit + agent overhead |
| vector_search | ~20s | Embed + retrieve + Claude generation |
| compare_videos | ~40s | Multi-chunk retrieval + Claude generation |

Agent overhead adds ~3-5s on top of raw tool latency (LLM reasoning about which tool to call).

**LangSmith fix:** Initial 403 errors caused by hitting US endpoint (`api.smith.langchain.com`) while account is EU-hosted. Fixed by adding `LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com` to `.env`. Traces now visible at `eu.smith.langchain.com` under the `askthevideo` project. Decision #65 logged.

**Import update:** `create_react_agent` from `langgraph.prebuilt` is deprecated. Production code uses `create_agent` from `langchain.agents` with `system_prompt=` parameter (renamed from `prompt=`). Function renamed to `create_askthevideo_agent` to avoid collision with the imported `create_agent`.

**Key decisions:**

| Decision | Choice | Reasoning |
|---|---|---|
| Agent framework | LangGraph ReAct via `create_agent` | Teacher's pattern, built-in tool routing + memory |
| Tool descriptions | Detailed with positive/negative examples | 100% routing accuracy on first try |
| System prompt | Explicit routing rules per tool | Removes ambiguity for borderline queries |
| Memory | MemorySaver (in-process) | Ephemeral by design, resets on container restart |
| LangSmith endpoint | EU (`eu.api.smith.langchain.com`) | Account hosted in EU, US endpoint returns 403 |

**Next:** Notebook 06 — Integration Flow (full pipeline: URL → transcript → chunks → embed → query → answer)